In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Files/v3_gitapress_final.csv")

df.columns

Index(['sa', 'hi', 'skr_out', 'syllable_count', 'skr_meter', 'meter_cd',
       'comments', 'prompt', 'split'],
      dtype='str')

In [4]:
df_train = df[df['split'] == 'train']
df_val = df[df['split'] == 'val']

df_train.shape, df_val.shape

((23346, 9), (2918, 9))

In [8]:
print("TRAIN SET")
df_train['meter_cd'].value_counts()

TRAIN SET


meter_cd
Anuṣṭubh            21766
Vasantatilakā         529
Śārdūlavikrīḍita      212
Indravajrā            174
Sragdharā             166
Vaṃśastha             139
Śikhariṇī             131
Upendravajrā           99
Mālinī                 75
Śālinī                 55
Name: count, dtype: int64

In [7]:
print("VAL SET")
df_val['meter_cd'].value_counts()

VAL SET


meter_cd
Anuṣṭubh            2721
Vasantatilakā         66
Śārdūlavikrīḍita      26
Indravajrā            22
Sragdharā             20
Vaṃśastha             17
Śikhariṇī             17
Upendravajrā          13
Mālinī                 9
Śālinī                 7
Name: count, dtype: int64

In [28]:
grouped = [
    {
        "meter": meter,
        "prompt": group["prompt"].drop_duplicates().tolist(),
        "syllable_count": group["syllable_count"].drop_duplicates().tolist()
    }
    for meter, group in df.groupby("meter_cd")
]

In [30]:
type(grouped), len(grouped)

(list, 10)

In [34]:
type(grouped[0]), type(grouped[0]), len(grouped[0]), grouped[0].keys()

(dict, dict, 3, dict_keys(['meter', 'prompt', 'syllable_count']))

In [ ]:
grouped[0]

{'meter': 'Anuṣṭubh',
 'prompt': ['Objective:\n\nGenerate a sanskrit verse in Anuṣṭubh meter based on a given meaning or theme while strictly following the metrical (?and rhyme) rules.\n\nRules of Chandas:\nSyllable Rules:\nLAGHU vowels: अ, इ, उ, ऋ, ऌ\nGURU vowels: आ, ई, ऊ, ॠ, ॡ, ए, ऐ, ओ, औ\nHRASVA vowels: अ, इ, उ, ऋ, ऌ\nDEERGHA vowels: आ, ई, ऊ, ॠ, ॡ, ए, ऐ, ओ, औ\nSyllable is marked laghu, guru and hrasva, deergha based on the vowel it contains.\nSyllable containing anusvāra("ं") or visarga("ः") is always marked as guru.\nSyllable that is followed by a conjunct consonant (saṁyuktākṣara)\nis always marked guru.\n\nThe goal is to generate Sanskrit verse that follows the anushtup meter rules for the given input translation.\nRULES:\nVerse Rules:\nThe verse contains 32 syllables/akshara and 4 padas in total.\nThe verse is divided in 2 lines, each containing 16 syllables.\nEach line is divided into 2 padas (quartets),each containing exactly 8 syllables.\nThe fifth syllable of every pada must

In [39]:
from tqdm import tqdm

new_df = pd.DataFrame(
    columns=["hi", "prompt", "meter_cd", "syllable_count", "split"]
)

for i, r in enumerate(tqdm(df_train.itertuples(index=False), total=len(df_train))):

    meter_index = i % 10
    meter_data = grouped[meter_index]

    sample_index = i // 10 % len(meter_data["prompt"])

    new_df.loc[i] = {
        "hi": r.hi,
        "prompt": meter_data["prompt"][sample_index],
        "meter_cd": meter_data["meter"],
        "syllable_count": meter_data["syllable_count"][sample_index],
        "split": "train"
    }

100%|██████████| 23346/23346 [23:38<00:00, 16.46it/s]


In [40]:
new_df.info()

<class 'pandas.DataFrame'>
Index: 23346 entries, 0 to 23345
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   hi              23346 non-null  str  
 1   prompt          23346 non-null  str  
 2   meter_cd        23346 non-null  str  
 3   syllable_count  23346 non-null  int64
 4   split           23346 non-null  str  
dtypes: int64(1), str(4)
memory usage: 39.2 MB


In [41]:
new_df['meter_cd'].value_counts()

meter_cd
Anuṣṭubh            2335
Indravajrā          2335
Mālinī              2335
Sragdharā           2335
Upendravajrā        2335
Vasantatilakā       2335
Vaṃśastha           2334
Śikhariṇī           2334
Śālinī              2334
Śārdūlavikrīḍita    2334
Name: count, dtype: int64

In [42]:
assert len(new_df) == len(df_train)
print(len(new_df))

23346


In [43]:
counts = new_df["meter_cd"].value_counts()

print(counts)
assert counts.max() - counts.min() <= 1

meter_cd
Anuṣṭubh            2335
Indravajrā          2335
Mālinī              2335
Sragdharā           2335
Upendravajrā        2335
Vasantatilakā       2335
Vaṃśastha           2334
Śikhariṇī           2334
Śālinī              2334
Śārdūlavikrīḍita    2334
Name: count, dtype: int64


In [44]:
print(new_df.isna().sum())
assert not new_df[["hi", "prompt", "meter_cd", "syllable_count"]].isna().any().any()

hi                0
prompt            0
meter_cd          0
syllable_count    0
split             0
dtype: int64


In [45]:
for meter in new_df["meter_cd"].unique():
    subset = new_df[new_df["meter_cd"] == meter]

    print(
        meter,
        "rows:", len(subset),
        "unique prompts:", subset["prompt"].nunique(),
        "syllables:", subset["syllable_count"].unique()
    )

Anuṣṭubh rows: 2335 unique prompts: 1 syllables: [32]
Indravajrā rows: 2335 unique prompts: 1 syllables: [44]
Mālinī rows: 2335 unique prompts: 1 syllables: [60]
Sragdharā rows: 2335 unique prompts: 1 syllables: [84]
Upendravajrā rows: 2335 unique prompts: 1 syllables: [44]
Vasantatilakā rows: 2335 unique prompts: 1 syllables: [56]
Vaṃśastha rows: 2334 unique prompts: 1 syllables: [48]
Śikhariṇī rows: 2334 unique prompts: 1 syllables: [68]
Śālinī rows: 2334 unique prompts: 1 syllables: [44]
Śārdūlavikrīḍita rows: 2334 unique prompts: 1 syllables: [76]


In [46]:
new_df.to_csv("Files/v3_gitapress_grpo_train_balanced.csv", index=False)